### Import libraries

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

### Load data

In [5]:
df = pd.read_csv(r"C:\Users\admin\Desktop\credit-risk-project\data\processed\credit_dashboard_data.csv")

df.head()

,SK_ID_CURR,TARGET,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AGE_YEARS,EMPLOYMENT_YEARS,CREDIT_INCOME_RATIO,ANNUITY_INCOME_RATIO,INCOME_SEGMENT,CREDIT_RISK_BAND,ANNUITY_RISK_BAND,NAME_EDUCATION_TYPE,OCCUPATION_TYPE,CODE_GENDER
0,100002,1,202500.0,406597.5,24700.5,25,1,2.007889,0.121978,High Income,Low Credit Burden,Low Repayment Burden,Secondary / secondary special,Laborers,M
1,100003,0,270000.0,1293502.5,35698.5,45,3,4.790750,0.132217,High Income,Medium Credit Burden,Low Repayment Burden,Higher education,Core staff,F
2,100004,0,67500.0,135000.0,6750.0,52,0,2.000000,0.100000,Low Income,Low Credit Burden,Low Repayment Burden,Secondary / secondary special,Laborers,M
3,100006,0,135000.0,312682.5,29686.5,52,8,2.316167,0.219900,Medium Income,Low Credit Burden,Medium Repayment Burden,Secondary / secondary special,Laborers,F
4,100007,0,121500.0,513000.0,21865.5,54,8,4.222222,0.179963,Medium Income,Medium Credit Burden,Medium Repayment Burden,Secondary / secondary special,Core staff,M


### Basic info

In [6]:
print("Shape:", df.shape)
print(df["TARGET"].value_counts())

Shape: (307511, 15)
TARGET
0    282686
1     24825
Name: count, dtype: int64


### Handle missing values

In [7]:
df = df.fillna("Unknown")

df.head()

,SK_ID_CURR,TARGET,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AGE_YEARS,EMPLOYMENT_YEARS,CREDIT_INCOME_RATIO,ANNUITY_INCOME_RATIO,INCOME_SEGMENT,CREDIT_RISK_BAND,ANNUITY_RISK_BAND,NAME_EDUCATION_TYPE,OCCUPATION_TYPE,CODE_GENDER
0,100002,1,202500.0,406597.5,24700.5,25,1,2.007889,0.121978,High Income,Low Credit Burden,Low Repayment Burden,Secondary / secondary special,Laborers,M
1,100003,0,270000.0,1293502.5,35698.5,45,3,4.790750,0.132217,High Income,Medium Credit Burden,Low Repayment Burden,Higher education,Core staff,F
2,100004,0,67500.0,135000.0,6750.0,52,0,2.000000,0.100000,Low Income,Low Credit Burden,Low Repayment Burden,Secondary / secondary special,Laborers,M
3,100006,0,135000.0,312682.5,29686.5,52,8,2.316167,0.219900,Medium Income,Low Credit Burden,Medium Repayment Burden,Secondary / secondary special,Laborers,F
4,100007,0,121500.0,513000.0,21865.5,54,8,4.222222,0.179963,Medium Income,Medium Credit Burden,Medium Repayment Burden,Secondary / secondary special,Core staff,M


### Encode categorical columns

In [8]:
df_model = df.copy()

for col in df_model.select_dtypes(include="object").columns:
    le = LabelEncoder()
    df_model[col] = le.fit_transform(df_model[col].astype(str))

df_model.head()

,SK_ID_CURR,TARGET,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AGE_YEARS,EMPLOYMENT_YEARS,CREDIT_INCOME_RATIO,ANNUITY_INCOME_RATIO,INCOME_SEGMENT,CREDIT_RISK_BAND,ANNUITY_RISK_BAND,NAME_EDUCATION_TYPE,OCCUPATION_TYPE,CODE_GENDER
0,100002,1,202500.0,406597.5,24700.5,25,1,2.007889,0.121978,0,1,1,4,8,1
1,100003,0,270000.0,1293502.5,35698.5,45,3,4.790750,0.132217,0,2,1,1,3,0
2,100004,0,67500.0,135000.0,6750.0,52,0,2.000000,0.100000,1,1,1,4,8,1
3,100006,0,135000.0,312682.5,29686.5,52,8,2.316167,0.219900,2,1,2,4,8,0
4,100007,0,121500.0,513000.0,21865.5,54,8,4.222222,0.179963,2,2,2,4,3,1


### Split features and target

In [9]:
X = df_model.drop("TARGET", axis=1)
y = df_model["TARGET"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (246008, 14)
Test shape: (61503, 14)


### Logistic Regression

In [11]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

lr_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=3000,
        class_weight="balanced",
        solver="lbfgs"
    ))
])

lr_pipeline.fit(X_train, y_train)

y_pred_lr = lr_pipeline.predict(X_test)
y_prob_lr = lr_pipeline.predict_proba(X_test)[:, 1]

In [12]:
print("LOGISTIC REGRESSION")

print("Accuracy:", accuracy_score(y_test, y_pred_lr))
print("Precision:", precision_score(y_test, y_pred_lr))
print("Recall:", recall_score(y_test, y_pred_lr))
print("F1:", f1_score(y_test, y_pred_lr))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_lr))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_lr))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_lr))

LOGISTIC REGRESSION
Accuracy: 0.5853698193584053
Precision: 0.1122356495468278
Recall: 0.5985901309164149
F1: 0.18902846239465734
ROC-AUC: 0.6227096057488556

Confusion Matrix:
[[33030 23508]
 [ 1993  2972]]

Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.58      0.72     56538
           1       0.11      0.60      0.19      4965

    accuracy                           0.59     61503
   macro avg       0.53      0.59      0.46     61503
weighted avg       0.88      0.59      0.68     61503



### Random Forest

In [15]:
# RANDOM FOREST WITH THRESHOLD TUNING

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    class_weight="balanced"
)

rf.fit(X_train, y_train)

# Probability score
y_prob_rf = rf.predict_proba(X_test)[:, 1]

# Change threshold here
threshold = 0.30

# Prediction using custom threshold
y_pred_rf = (y_prob_rf >= threshold).astype(int)

print("RANDOM FOREST WITH THRESHOLD:", threshold)

print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print("Precision:", precision_score(y_test, y_pred_rf))
print("Recall:", recall_score(y_test, y_pred_rf))
print("F1:", f1_score(y_test, y_pred_rf))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_rf))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf))

RANDOM FOREST WITH THRESHOLD: 0.3
Accuracy: 0.1832918719412061
Precision: 0.08688509628547961
Recall: 0.9587109768378651
F1: 0.1593305439330544
ROC-AUC: 0.6499210024310753

Confusion Matrix:
[[ 6513 50025]
 [  205  4760]]

Classification Report:
              precision    recall  f1-score   support

           0       0.97      0.12      0.21     56538
           1       0.09      0.96      0.16      4965

    accuracy                           0.18     61503
   macro avg       0.53      0.54      0.18     61503
weighted avg       0.90      0.18      0.20     61503



In [16]:
import numpy as np
from sklearn.metrics import f1_score

thresholds = np.arange(0.3, 0.7, 0.05)

best_threshold = 0
best_f1 = 0

for t in thresholds:
    y_pred_temp = (y_prob_rf >= t).astype(int)
    f1 = f1_score(y_test, y_pred_temp)
    
    print(f"Threshold: {t:.2f} | F1: {f1:.4f}")
    
    if f1 > best_f1:
        best_f1 = f1
        best_threshold = t

print("\nBest Threshold:", best_threshold)

Threshold: 0.30 | F1: 0.1593
Threshold: 0.35 | F1: 0.1661
Threshold: 0.40 | F1: 0.1793
Threshold: 0.45 | F1: 0.1954
Threshold: 0.50 | F1: 0.2059
Threshold: 0.55 | F1: 0.2125
Threshold: 0.60 | F1: 0.2057
Threshold: 0.65 | F1: 0.1658

Best Threshold: 0.5499999999999999


In [17]:
# FINAL RANDOM FOREST MODEL

threshold = 0.55

y_pred_final = (y_prob_rf >= threshold).astype(int)

print("FINAL RANDOM FOREST MODEL")
print("Threshold:", threshold)

print("Accuracy:", accuracy_score(y_test, y_pred_final))
print("Precision:", precision_score(y_test, y_pred_final))
print("Recall:", recall_score(y_test, y_pred_final))
print("F1:", f1_score(y_test, y_pred_final))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_rf))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_final))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_final))

FINAL RANDOM FOREST MODEL
Threshold: 0.55
Accuracy: 0.7561582361836008
Precision: 0.1436993891177724
Recall: 0.40745216515609267
F1: 0.21246652313185949
ROC-AUC: 0.6499210024310753

Confusion Matrix:
[[44483 12055]
 [ 2942  2023]]

Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.79      0.86     56538
           1       0.14      0.41      0.21      4965

    accuracy                           0.76     61503
   macro avg       0.54      0.60      0.53     61503
weighted avg       0.87      0.76      0.80     61503



In [14]:
print("RANDOM FOREST")

print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print("Precision:", precision_score(y_test, y_pred_rf))
print("Recall:", recall_score(y_test, y_pred_rf))
print("F1:", f1_score(y_test, y_pred_rf))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_rf))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf))

RANDOM FOREST
Accuracy: 0.6624229712371754
Precision: 0.12709503800575989
Recall: 0.5421953675730111
F1: 0.20592059970932455
ROC-AUC: 0.6499210024310753

Confusion Matrix:
[[38049 18489]
 [ 2273  2692]]

Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.67      0.79     56538
           1       0.13      0.54      0.21      4965

    accuracy                           0.66     61503
   macro avg       0.54      0.61      0.50     61503
weighted avg       0.88      0.66      0.74     61503



In [18]:
# Feature Importance - Random Forest

feature_importance = pd.Series(
    rf.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

print("Top 10 Important Features:")
print(feature_importance.head(10))

Top 10 Important Features:
AGE_YEARS               0.163967
EMPLOYMENT_YEARS        0.141647
AMT_CREDIT              0.109160
NAME_EDUCATION_TYPE     0.094682
AMT_ANNUITY             0.092615
CREDIT_INCOME_RATIO     0.079849
ANNUITY_INCOME_RATIO    0.079699
SK_ID_CURR              0.061010
CODE_GENDER             0.058382
AMT_INCOME_TOTAL        0.048743
dtype: float64


In [19]:
y_pred_final = (y_prob_rf >= threshold).astype(int)

In [20]:
# Create output dataframe

output = X_test.copy()

output["Actual_TARGET"] = y_test.values
output["Predicted_TARGET"] = y_pred_final
output["Default_Probability"] = y_prob_rf

# Save to CSV
output.to_csv("data/processed/ml_predictions.csv", index=False)

print("✅ Predictions saved successfully!")

OSError: Cannot save file into a non-existent directory: 'data\processed'

In [22]:
import os

# Create folder if not exists
os.makedirs("data/processed", exist_ok=True)

# Save file
output.to_csv("data/processed/ml_predictions.csv", index=False)

print("✅ Predictions saved successfully!")

✅ Predictions saved successfully!


In [23]:
import os
print(os.getcwd())

c:\Users\admin\Desktop\credit-risk-project\src


In [24]:
import os

for root, dirs, files in os.walk("."):
    for file in files:
        if "ml_predictions" in file:
            print(os.path.join(root, file))

.\data\processed\ml_predictions.csv


In [25]:
import os

os.makedirs("../data/processed", exist_ok=True)

output.to_csv("../data/processed/ml_predictions.csv", index=False)

print("✅ Saved in main data folder")

✅ Saved in main data folder
